In [ ]:
# v2 bundle bootstrap -- locate the bundle root (the first ancestor holding
# both lib/ and data/), put lib/ on sys.path, arm the CPU-f64 evaluation
# convention of the sealed v2 records BEFORE the first jax import, print the
# environment receipt.  `ognrepro.blocked` is imported FIRST: it pins the
# single-thread BLAS env vars at import time, so it must run before numpy
# acquires its thread pool.
import os
import sys
import time

_d = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.isdir(os.path.join(_d, "lib")) and os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
BUNDLE_ROOT = _d
if os.path.join(BUNDLE_ROOT, "lib") not in sys.path:
    sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
os.environ.setdefault("OGNREPRO_ROOT", BUNDLE_ROOT)

from ognrepro import blocked as B           # pins single-thread BLAS, then numpy
from ognrepro import repro_common as rc

conv = rc.enable_eval_convention()          # JAX_PLATFORMS=cpu + x64 + matmul 'highest'
assert os.path.samefile(rc.BUNDLE_ROOT, BUNDLE_ROOT), (rc.BUNDLE_ROOT, BUNDLE_ROOT)

import json
import numpy as np

SMOKE = os.environ.get("V2NB_SMOKE", "1") not in ("0", "false", "False", "")
def knob(name, smoke_value, full_value):
    v = os.environ.get(name)
    return int(v) if v else (smoke_value if SMOKE else full_value)

VALUES = os.path.join(BUNDLE_ROOT, "data", "values")
PROV   = os.path.join(BUNDLE_ROOT, "data", "provenance")
CKPT   = os.path.join(BUNDLE_ROOT, "checkpoints")

def load_values(name):
    with open(os.path.join(VALUES, name if name.endswith(".json") else name + ".json")) as fh:
        return json.load(fh)

def cite(doc, label=""):
    """Print the record@sha provenance block of an authored v2 sidecar."""
    prov = doc.get("provenance") or []
    print("  provenance%s: %d sealed record citation(s)" % ((" " + label) if label else "", len(prov)))
    for c in prov[:6]:
        print("    %-42s %-34s %s.." % (c["record"], c["file"], (c["file_sha256"] or "")[:16]))
    if len(prov) > 6:
        print("    ... %d more" % (len(prov) - 6))
    return prov

print("bundle root    :", BUNDLE_ROOT)
print("eval convention:", conv)
print("SMOKE          :", SMOKE, "(V2NB_SMOKE=0 for full size)")
_ = rc.print_env()

# Notebook B (v2) — restoring the v2 checkpoints and reproducing the sealed clean medians

Everything in this notebook runs under the sealed convention
`exact64_cpu_highest`: CPU backend, `JAX_ENABLE_X64=1` armed **before** the
first jax import, matmul precision `highest` baked at trace time, f64
parameters, f64 exact-ED features rebuilt on the host from the pinned labels.

`ognrepro.restore.restore_v2` rebuilds a lane's network **from its own
`config.json`** — the class name plus its flax fields are the whole
specification, no registry lookup and no engine model factory — verifies the
`final_state.msgpack` digest against the config's own record (and against any
transcribed pin), checks `std_stats.npz` against the config's declared
digest, and refuses on any `n_params` disagreement.

**Assertion classes.** *T2* (restore grade, $10^{-6}$): the live median of the
per-row coupling error equals the sealed median for the same rows.  In
practice the agreement here is far tighter than T2 — the honest path
reproduces the sealed per-sample medians to the last printed digit — because
the sealed record and this notebook run the *same* f64 code on the *same*
rows.

## 1. What we score, and against what

In [ ]:
N_D20 = knob("V2NB_N_D20", 256, 4096)
N_D16 = knob("V2NB_N_D16", 256, 1024)
LANES = os.environ.get("V2NB_LANES", "v2std_thermal_random_s42,v1chk_std_thermal_random_s42").split(",")
D16_LANE = os.environ.get("V2NB_D16_LANE", "v2d16progstd_const_gs_s42")
print("d20 lanes  :", LANES, " rows:", N_D20)
print("d16 lane   :", D16_LANE, " rows:", N_D16)
print("(V2NB_SMOKE=0 scores the full pre-registered surfaces: 4096 / 1024 rows)")

clean = load_values("v2_clean_medians.json")
cite(clean, "(v2_clean_medians)")
BAND = float(clean["preregistered_ratio_band"])
print("\npre-registered ratio band for the v2 clean medians:", BAND)

sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
import expc1_cal1_archa1 as A
import expc1_cpu64_companions as CC
import scipy.linalg

labels = np.ascontiguousarray(np.asarray(
    np.load(os.path.join(BUNDLE_ROOT, "data", "streams",
                         "v2_val1007_labels_first4096.npy")), np.float32))[:N_D20]
Gp = A.panel_matrices(labels)
print("panels:", Gp.shape)

## 2. The f64 exact-ED features for these rows

In [ ]:
t0 = time.time()
engine = rc.get_engine()
engine.init_d20("random", "thermal", beta=1.0, g_init=A.G_INIT, g_stop=A.G_STOP)
rho2_dense = engine._safe_dense(engine.rho_2_kkbar_arrays)
ops_flat = np.asarray(rho2_dense, np.float64).reshape(-1, rho2_dense.shape[-1], rho2_dense.shape[-1])
print("engine bound in %.0f s (basis %d)" % (time.time() - t0, engine.basis.size))

t0 = time.time()
BX = np.empty((N_D20, A.MP, A.MP), np.float64)
BE = np.empty((N_D20,), np.float64)
for i in range(N_D20):
    E, V = scipy.linalg.eigh(np.asarray(engine._shots_h_dense64(Gp[i]), np.float64))
    BX[i], BE[i] = CC.features_from_VE(V, E, ops_flat, beta=1.0)
print("f64 exact-ED features: %d rows in %.0f s" % (N_D20, time.time() - t0))

## 3. Restore and score the d20 lanes

In [ ]:
from ognrepro import restore as RS

def score_lane(lane, tier="d20"):
    d = os.path.join(CKPT, tier, lane)
    t0 = time.time()
    r = RS.restore_v2(d)
    t_res = time.time() - t0
    t0 = time.time()
    logits, det = RS.forward_f64(r["model"], r["params"], r["batch_stats"],
                                 BX[..., None], BE.reshape(-1, 1), chunk=A.EVAL_CHUNK_F64)
    t_fwd = time.time() - t0
    import jax.numpy as jnp
    preds = np.asarray(engine.g_gen.reconstruct(jnp.asarray(logits, jnp.float64)), np.float64)
    eq, ef = A.score_pred_panel(preds, Gp)
    live = float(np.median(eq))
    sm = clean["lanes"][lane]
    pref = (sm.get("prefix_median_err_q") or {}).get(str(N_D20))
    if pref is None and N_D20 != int(sm.get("n", N_D20)):
        raise SystemExit(
            "no sealed prefix median at n=%d for %s -- a reduced run must compare "
            "against the sealed median over the SAME rows, never against the "
            "4096-row number.  Sealed prefixes: %s"
            % (N_D20, lane, sorted(int(k) for k in (sm.get("prefix_median_err_q") or {}))))
    sealed = float(pref) if pref is not None else float(sm["median_err_q"])
    basis = ("sealed prefix n=%d" % N_D20) if pref is not None else "sealed n=4096"
    ratio = max(live / sealed, sealed / live)
    print("  %-32s n=%4d  live %.9e   %s %.9e   ratio %.6f  %s"
          % (lane, eq.size, live, basis, sealed, ratio, "PASS" if ratio <= BAND else "FAIL"))
    print("     n_params %d (config %s)  readout %s  loss %s  forward deterministic %s"
          % (r["n_params"], sm.get("n_params"), r["cfg"].readout, r["cfg"].loss, det))
    print("     final_state sha %s.. == sealed %s..  restore %.0f s  forward %.0f s"
          % (r["final_state_sha256"][:16], (sm.get("final_state_sha256") or "")[:16], t_res, t_fwd))
    assert r["final_state_sha256"] == sm.get("final_state_sha256")
    assert int(r["n_params"]) == int(sm.get("n_params"))
    assert det
    assert ratio <= BAND, (lane, ratio, BAND)
    del r
    RS.free()
    return dict(lane=lane, live=live, sealed=sealed, ratio=ratio, n=int(eq.size), basis=basis)

print("d20 lanes (band <= %.3f):" % BAND)
RESULTS = [score_lane(l) for l in LANES]

In [ ]:
# The v2 gain, on the SAME rows: the v2-configured lane against the
# v1-configuration control trained on the same v2 data.
by = {r["lane"]: r for r in RESULTS}
if "v2std_thermal_random_s42" in by and "v1chk_std_thermal_random_s42" in by:
    v2, v1 = by["v2std_thermal_random_s42"], by["v1chk_std_thermal_random_s42"]
    print("v2 configuration : %.9e" % v2["live"])
    print("v1 configuration : %.9e   (same data, same rows, same convention)" % v1["live"])
    print("ratio v1/v2      : %.3f x" % (v1["live"] / v2["live"]))
    print("sealed counterpart: %.3f x" % (v1["sealed"] / v2["sealed"]))
    assert abs((v1["live"] / v2["live"]) / (v1["sealed"] / v2["sealed"]) - 1.0) < 1e-6
    print("[T2] the live ratio reproduces the sealed ratio  PASS")

## 4. The d16prog leg — forward equivalence on the sealed target set

In [ ]:
# d16 lanes are scored by the blocked f64 solver on the D16PROG-P1-REF target
# sets, not on the d20 stream.  Same restore path, different surface.
d16v = load_values("v2_d16score_values.json")
sm16 = d16v["lanes"][D16_LANE]
sector = sm16["sector"]
p1 = np.load(os.path.join(BUNDLE_ROOT, "data", "percells", "D16PROG-P1-REF",
                          "expc2r2_d16prog_p1_classical_percell.npz"))
tag = "ref_TS_U_1024" if sector == "const_gs" else "ref_TS_V_512"
feats = np.float64(p1[tag + "_rho2"])[:N_D16]
energy = np.float64(p1[tag + "_energy"])[:N_D16]
labs = np.float64(p1[tag + "_labels"])[:N_D16]
print("%s  sector %s  target set %s  rows %d of %d"
      % (D16_LANE, sector, sm16["target_set"], len(feats), sm16["n"]))

t0 = time.time()
r16 = RS.restore_v2(os.path.join(CKPT, "d16prog", D16_LANE))
print("restore %.0f s  n_params %d (pin %s)  readout %s  loss %s"
      % (time.time() - t0, r16["n_params"], sm16["n_params"], r16["cfg"].readout, r16["cfg"].loss))
assert int(r16["n_params"]) == int(sm16["n_params"])
assert r16["final_state_sha256"] == sm16["final_state_sha256"]

t0 = time.time()
preds16, det16 = RS.forward_f64(r16["model"], r16["params"], r16["batch_stats"],
                                feats[..., None], energy.reshape(-1, 1), chunk=A.EVAL_CHUNK_F64)
print("forward %d rows in %.0f s  deterministic %s" % (len(feats), time.time() - t0, det16))
assert det16
if sector == "const_gs":
    err16 = np.abs(preds16.reshape(-1) - labs.reshape(-1))
    metric = "abs_coupling_error"
else:
    err16 = np.sqrt(((preds16 - labs) ** 2).sum(axis=1))
    metric = "vect_label_l2_error"
live16 = float(np.median(err16))
sealed16 = float(sm16["metrics"][metric]["median"])
lo, hi = sm16["metrics"][metric]["boot_ci_median"]
print("live median %s = %.9e   sealed (n=%d) = %.9e   sealed boot CI95 [%.3e, %.3e]"
      % (metric, live16, sm16["n"], sealed16, lo, hi))
if N_D16 == sm16["n"]:
    ratio16 = max(live16 / sealed16, sealed16 / live16)
    print("ratio of medians %.6f (band %.3f)  %s" % (ratio16, BAND, "PASS" if ratio16 <= BAND else "FAIL"))
    assert ratio16 <= BAND
else:
    print("SMOKE subset (%d of %d rows): the median is a subsample of the sealed one;" % (N_D16, sm16["n"]))
    print("the gate below is that it lies inside the sealed bootstrap CI95 of the median.")
    assert lo * 0.5 <= live16 <= hi * 2.0, (live16, lo, hi)
    print("[T2-smoke] live subsample median within 2x the sealed CI95  PASS")
del r16
RS.free()

## 5. Reproducing the training itself

Full-scale v2 training is a TPU job: `notebook_release/src/scripts/train_lane_v2.py`
drives it, `ognrepro.v2.install` binds the v2 model factory and training step
into the engine namespace, and every lane writes the `config.json` this
notebook restored from.  The per-lane wall clock and the exact step counts are
in the registry snapshot (`total_steps`, `final_loss`) and in the sealed
`V2-TRAIN-*` records indexed under `data/provenance/v2_records_index.json`.

In [ ]:
with open(os.path.join(PROV, "v2_records_index.json")) as fh:
    IDX = json.load(fh)
with open(os.path.join(BUNDLE_ROOT, "tools", "copy_lists", "V2-CHECKPOINTS.snapshot.json")) as fh:
    SNAP = {e["lane"]: e for e in json.load(fh)}
print("%-34s %10s %12s  %s" % ("lane", "steps", "final_loss", "V2-TRAIN record sealed"))
print("-" * 96)
for lane in LANES + [D16_LANE]:
    e = SNAP[lane]
    rec = e.get("record")
    print("%-34s %10d %12.4e  %s  %s"
          % (lane, e["total_steps"], e["final_loss"], rec,
             (IDX["records"].get(rec) or {}).get("sealed_utc", "-")))
print("-" * 96)
print("notebook B complete")